<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/ch07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 指示に従うためのファインチューニング
- LLMのインストラクションファインチューニングプロセス
- インストラクションチューニングされたLLMを評価する

## 7.2 教師ありインストラクションチューニングのためのデータセット準備

In [ ]:
# データセットをダウンロードする
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r") as file:
        data = json.load(file)

    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print(f"Number of entries: {len(data)}")

In [ ]:
# サンプルエントリを確認
print(f"Example entry:\n {data[50]}")

In [ ]:
# 別のサンプルも確認
# inputは空の場合もある
print(f"Another example entry:\n {data[999]}")

In [ ]:
# プロンプトフォーマット関数を実装する
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

In [ ]:
# サンプルテキストで確認
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

In [ ]:
# inputフィールドが空のサンプルの場合
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

In [ ]:
# データセットを分割する
train_portion = int(len(data) * 0.85) # 85%のデータを学習に使用
test_portion = int(len(data) * 0.1) # 10%のデータをテストに使用
val_portion = len(data) - train_portion - test_portion # 残り5%のデータを検証用に使用

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"Training set length: {len(train_data)}")
print(f"Test set length: {len(test_data)}")
print(f"Validation set length: {len(val_data)}")

## 7.3 データを訓練バッチにまとめる
- サンプルデータを同じ長さに効率良くパディングする方法
- サンプルのリストをバッチにまとめるためにデフォルトのcollate関数を使用する
- collate関数は個々のデータサンプルのリストを1つのバッチにまとめて、訓練中にモデルが効率よく処理できるようにする役割を果たす

In [ ]:
# 指示データセットクラスを実装
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            # テキストを事前トークン化
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
# '<|endoftext|>'を使ってパディングするためにトークンIDを取得
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))